In [2]:
import pandas as pd
import numpy as np

Data loading and data parsing

In [3]:
df = pd.read_csv("Data/tanzania.csv")

In [4]:
#Adding a Country column
df["Country"] = "tanzania"

In [5]:
#Convering YEAR and DOY into a proper datetime column
df["date"] = pd.to_datetime(df["YEAR"] * 1000 + df["DOY"], format="%Y%j")

In [6]:
#Extract Month as a separate column for seasonal analysis
df["month"] = df["date"].dt.month

In [ ]:
#checking if the changes re actually made
df[["YEAR", "DOY", "date", "month", "Country"]].head()

Summary Statistics & Missing-Value Report

In [ ]:
#Replace -999 with NaN (Do this first!)
df.replace(-999, np.nan, inplace=True)

In [ ]:
#Checking for duplicates
duplicate_count = df.duplicated().sum()
print(f"Total duplicate rows found: {duplicate_count}")

In [10]:
###Drop duplicates if any exist
if duplicate_count > 0:
    df.drop_duplicates(inplace=True)
    print("Duplicates removed.")

In [ ]:
#descriptive statistics for all numeric columns
df.describe()

This section gives as the average,minumum,maximum,std,and interquatile values... of each column giving as a general understandng of the data. For instance the mean daily air temperature at 2 meters above the surface is 26.8 degrees as observed in the output.

In [12]:
#Compute null counts and percentages
null_counts = df.isna().sum()
null_percentages = (null_counts / len(df)) * 100

# Create a summary table
missing_report = pd.DataFrame({
    'Null Count': null_counts,
    'Percentage (%)': null_percentages
    })

In [ ]:
# Display only columns with missing data
print("Missing Data Report:")
print(missing_report[missing_report['Null Count'] > 0])

In [ ]:
# List columns with > 5% nulls
high_nulls = null_percentages[null_percentages > 5]
if not high_nulls.empty:
    print("\nColumns with > 5% Missing Data:")
    for col, val in high_nulls.items():
        print(f"- {col}: {val:.2f}%")
else:
    print("\nNo columns exceed 5% missing data.")

Since no columns exceed the 5% null threshold, we can proceed with full confidence to our statistial analysis work.

Outlier Detection & Basic Cleaning

In [ ]:
#Outlier Detection & Basic Cleaning
from scipy import stats

# Columns to check for outliers
weather_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX']

# Calculate absolute Z-scores
z_scores = np.abs(stats.zscore(df[weather_cols]))

# Create a boolean mask where any column has |Z| > 3
outlier_mask = (z_scores > 3).any(axis=1)
outlier_count = outlier_mask.sum()

print(f"Total rows flagged as outliers (|Z| > 3): {outlier_count}")

It would be better to keep the outliers since these extreme values tells us a sudden spike in the weather condition and is no an error made by measurement.And this extreme weather conditions are naturally expected.

In [16]:
#List of all weather-related variables to be filled
weather_variables = [
    'T2M', 'T2M_MAX', 'T2M_MIN', 'T2M_RANGE', 
    'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX', 
    'PS', 'QV2M'
]

In [ ]:
#Drop rows with >30% missing values
# (If a row has less than 70% of the columns filled, we discard it as unreliable)
initial_rows = len(df)
df.dropna(thresh=int(len(df.columns) * 0.7), inplace=True)
dropped_rows = initial_rows - len(df)

#  Apply forward-fill to the specific weather columns
# This propagates the last valid observation forward to next valid
df[weather_variables] = df[weather_variables].ffill()

print(f"Cleaning complete.")
print(f"Rows dropped (threshold < 70%): {dropped_rows}")

In [ ]:
df.to_csv('Data/tanzania_clean.csv', index=False)
print("Data saved to Data/tanzania_clean.csv")

Time Series Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Aggregate data to monthly level
monthly_df = df.groupby(['YEAR', 'month']).agg({
    'T2M': 'mean', 
    'PRECTOTCORR': 'sum'
}).reset_index()

# 2. CREATE A TRUE DATETIME COLUMN (This fixes the error)
monthly_df['Plot_Date'] = pd.to_datetime(monthly_df[['YEAR', 'month']].assign(day=1))

# Find coordinates for annotations using the new Plot_Date
warmest = monthly_df.loc[monthly_df['T2M'].idxmax()]
coolest = monthly_df.loc[monthly_df['T2M'].idxmin()]

# 3. Plotting with true dates
plt.figure(figsize=(15, 6))
sns.lineplot(data=monthly_df, x='Plot_Date', y='T2M', marker='o', color='firebrick', linewidth=2)

# Annotations use the Plot_Date
plt.annotate(f"Warmest: {warmest['T2M']:.2f}°C", 
             xy=(warmest['Plot_Date'], warmest['T2M']), xytext=(10, 10),
             textcoords='offset points', arrowprops=dict(arrowstyle='->'))

plt.annotate(f"Coolest: {coolest['T2M']:.2f}°C", 
             xy=(coolest['Plot_Date'], coolest['T2M']), xytext=(10, -20),
             textcoords='offset points', arrowprops=dict(arrowstyle='->'))

plt.title("Monthly Average Temperature (T2M) in Tanzania (2015-2026)", fontsize=14)
plt.xlabel("Year")
plt.ylabel("Average Temperature (°C)")
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Find the peak rainy month
peak_rain = monthly_df.loc[monthly_df['PRECTOTCORR'].idxmax()]

# Plotting using Plot_Date
plt.figure(figsize=(15, 6))
# Using a fixed width to make bars look nice with dates (approx 20 days wide)
plt.bar(monthly_df['Plot_Date'], monthly_df['PRECTOTCORR'], color='royalblue', alpha=0.8, width=20)

plt.annotate(f"Peak: {peak_rain['PRECTOTCORR']:.1f} mm", 
             xy=(peak_rain['Plot_Date'], peak_rain['PRECTOTCORR']), xytext=(0, 10),
             textcoords='offset points', ha='center', fontweight='bold',
             arrowprops=dict(arrowstyle='->'))

plt.title("Monthly Total Precipitation (PRECTOTCORR) in Tanzania (2015-2026)", fontsize=14)
plt.xlabel("Year")
plt.ylabel("Total Rainfall (mm)")
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In the test series analysis,the warmest average monthly tempreture recorded is 28.99 degrees while the coolest that is recorded is 24.27. In the monthly precipitation; the highest peak rain is 482.3mm.

In [ ]:
#correltion 
# Select only numeric columns for correlation
numeric_cols = df.select_dtypes(include=[np.number])

plt.figure(figsize=(12, 8))
correlation_matrix = numeric_cols.corr()

# Create the heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='RdBu_r', fmt=".2f", center=0)

plt.title("Correlation Heatmap of Climate Variables (Tanzania)", fontsize=14)
plt.show()

The 3 strongest correlations observed
    1.T2M and T2M_MIN (Positive Correlation: 0.95)
A correlation of 0.95 indicates that the daily mean temperature is almost entirely driven by the nightime/minimum temperatures.

    2.T2M_MIN and QV2M(positive Correlation: 0.89)
This shows that Minimum daily temperature at 2 meters and Specific humidity - mass of water vapor per unit mass of moist air are strongly related were one affects the other positively.
    
    3.WS2M and WS2M_MAX(positive correlation:0.95)

This shows that Mean daily wind speed at 2 meters and Maximum daily wind speed at 2 meters have high correlation. This shows that the maximum wind speed will affect the mean i.e when max wind speed is high the mean will also be high. And this is expected based on basic maths concept for calculating mean.

In [ ]:
#sactter plots
# Create a figure with two subplots side-by-side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Temperature vs Relative Humidity
sns.scatterplot(data=df, x='T2M', y='RH2M', alpha=0.3, color='teal', ax=ax1)
ax1.set_title("Relationship: Temperature vs. Humidity")
ax1.set_xlabel("Mean Temp (°C)")
ax1.set_ylabel("Relative Humidity (%)")

# Plot 2: Temp Range vs Wind Speed
sns.scatterplot(data=df, x='T2M_RANGE', y='WS2M', alpha=0.3, color='coral', ax=ax2)
ax2.set_title("Relationship: Temp Range vs. Wind Speed")
ax2.set_xlabel("Daily Temp Range (°C)")
ax2.set_ylabel("Wind Speed (m/s)")

plt.tight_layout()
plt.show()

Distribution Analysis

In [ ]:
#plotting histogram
plt.figure(figsize=(10, 6))

# Histogram with Log Scale on the Y-axis to handle skewness
sns.histplot(df['PRECTOTCORR'], bins=30, color='royalblue', kde=True)
plt.yscale('log')  # Apply log scale

plt.title("Distribution of Daily Precipitation (Log Scale)", fontsize=14)
plt.xlabel("Precipitation (mm/day)")
plt.ylabel("Frequency (Log Count)")
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(12, 7))

# We use a subset or alpha to prevent the chart from being too crowded
# Size is scaled by PRECTOTCORR (multiplied by 5 to make bubbles visible)
scatter = plt.scatter(df['T2M'], df['RH2M'], 
                      s=df['PRECTOTCORR'] * 5, 
                      c=df['PRECTOTCORR'], 
                      cmap='Blues', alpha=0.4, edgecolors='grey')

# Add a colorbar to show rain intensity
plt.colorbar(scatter, label='Precipitation (mm/day)')

plt.title("Bubble Chart: Temperature vs. Humidity (Size = Rain Intensity)", fontsize=14)
plt.xlabel("Mean Temperature (°C)")
plt.ylabel("Relative Humidity (%)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()